In [ ]:
from wekeo_frp_l3.download import get_FRP_products
from wekeo_frp_l3.reader import read_FRP_product
import xarray as xr

from wekeo_frp_l3.stubs import log
from wekeo_frp_l3.stubs import env
from datetime import date
    
def  get_log_event(day: date) -> xr.Dataset:
    """Download and Compile multiple FRP files into a single xarray Dataset."""
    
    log.info(f"Downloading FRP products for date: {day}")
    
    eday = day.strftime("%Y-%m-%d")
    sday = day.strftime("%Y-%m-%d") # same day for single day query

    files = get_FRP_products(
        start_date = sday,
        end_date = eday,
    )

    # Read specific variables
    variables = [
        'latitude', 
        'longitude',
        'FRP_SWIR',
        'FRP_uncertainty_SWIR',
        'FRP_MWIR', 
        'FRP_uncertainty_MWIR',
        'confidence_SWIR_SAA',  # SAA flag for quality control
        'solar_zenith',
        # 'S8_Fire_pixel_BT',  # This is from a different dimension
    ]
    
    log.info(f"Compiling {len(files)} FRP products into a Log Event dataset")
    datasets = []
    
    for f in files:
        ds = read_FRP_product(f, variables=variables).compute()
        datasets.append(ds)
        
    log_event = xr.concat(datasets, dim='merged_MWIR1kmStandard_SWIR1km')
    log_event = log_event.rename_dims({"merged_MWIR1kmStandard_SWIR1km": "nb_detection"})
    log_event = log_event.rename_vars({"solar_zenith": "sza"})
    
    log_event.attrs['source_files'] = [str(f.name) for f in files]
    log_event.attrs['date'] = eday
    
    return log_event

In [ ]:
# Download and compile daily log event

# day = date(2022, 7, 15)
day = date(2024, 7, 15)

log_event = get_log_event(day)
print(log_event)

In [ ]:
# Save L3 result (non gridded)
output_dir = env.getdir("OUTPUT_DIR")
l3_output_file = f"{output_dir}/log_event_{day.strftime('%Y_%m_%d')}.nc"
log_event.to_netcdf(l3_output_file, mode="w")

In [ ]:
# Compute L3 gridded data

from wekeo_frp_l3.log_event_accumulator import accumulate_events_to_grid
from wekeo_frp_l3.plot_L3_FRP import plot_L3_FRP

from pathlib import Path
import xarray as xr

log_event = xr.open_dataset(l3_output_file)

# compute the reprojected L3 dataset
PROPER_RESOLUTION_WIDTH = 3272  # ~1km at the equator, adjust as needed
SIMPLER_RESOLUTION_WIDTH = 600  # for demo purposes
gl3 = accumulate_events_to_grid(log_event, width=SIMPLER_RESOLUTION_WIDTH, min_count=1) # gridded level3

# save the file
gridded_l3_output_file = f"{output_dir}/FRP_l3_{day.strftime('%Y_%m_%d')}.nc"
gl3.to_netcdf(gridded_l3_output_file, mode="w")


Availables = [ 'FRP_SWIR', 'FRP_SWIR_no_SAA', 'FRP_MWIR' ]
var = 'FRP_SWIR_no_SAA'
colormap = "plasma"
figsize = (16 * 2, 10 * 2)

figdir = Path(f"{output_dir}/figures")
figdir.mkdir(parents=False, exist_ok=True)
# figdir = None  # Disables figure saving

# Single variable
plot_L3_FRP(gl3, variable=f'{var}_mean',  cmap=colormap, figsize=figsize, save_fig_dir=figdir)
plot_L3_FRP(gl3, variable=f'{var}_min',   cmap=colormap, figsize=figsize, save_fig_dir=figdir)
plot_L3_FRP(gl3, variable=f'{var}_max',   cmap=colormap, figsize=figsize, save_fig_dir=figdir)
plot_L3_FRP(gl3, variable=f'{var}_std',   cmap=colormap, figsize=figsize, save_fig_dir=figdir)
plot_L3_FRP(gl3, variable=f'{var}_count', cmap='autumn', figsize=figsize, save_fig_dir=figdir)
